Q1: Build Your Personalized Knowledge Base: Take your college roll number. Extract its digits. Build a pandas
DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your
own roll number digits as follows:
• Take the LAST TWO DIGITS of your roll number. For each digit d, compute category = ["billing", "account",
"general"][d % 3]. Invent one realistic question+answer+3 keywords per entry that fits the assigned category (e.g.
if d%3 gives "account", write a question like “how do I update my registered mobile number”).
• # Example roll number ...23 -> digits 2, 3
• # digit 2 -> category[2 % 3] = general
• # digit 3 -> category[3 % 3] = billing

In [9]:
import pandas as pd

roll_num=input("enter your roll number: ")
last2=[roll_num[-2:]]
categories=["billing","account","general"]

fixed_entries = [
 {"question": "what is the annual fee",
  "answer": "The annual fee is Rs 500.",
 "keywords": "fee cost price charge",
  "category": "billing"},

 {"question": "how to reset password",
  "answer": "Go to Settings > Reset Password.",
 "keywords": "password reset login",
  "category": "account"},

 {"question": "what are your working hours",
  "answer": "We are open 9 AM to 5 PM.",
 "keywords": "hours timing open time",
  "category": "general"},

 {"question": "how can i pay the fee",
  "answer": "You can pay via UPI, card, or net banking.",
 "keywords": "pay payment upi fee",
  "category": "billing"},
]

new_entries=[]
for digit in last2:
  d=int(digit)
  category=categories[d%3];

  if category == "billing":
        question = "how can i check my fee payment status"
        answer = "You can check your fee payment status in the payment section."
        keywords = "fee payment status billing"

  elif category == "account":
        question = "how do i update my registered mobile number"
        answer = "You can update your registered mobile number from account settings."
        keywords = "mobile number account update"

  else:
        question = "what documents are required for registration"
        answer = "You need your college ID and required registration documents."
        keywords = "documents registration general"

  new_entries.append({
    "question":question,
    "answer":answer,
    "keywords":keywords,
    "category":category
  })

all_entries=fixed_entries+new_entries
df=pd.DataFrame(all_entries)
df

enter your roll number: 1024170288


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my registered mobile number,You can update your registered mobile number f...,mobile number account update,account


Q2. Generate and Score a Hypothesis - Implement a scoring function that takes a query string and returns all matching entries ranked by confidence

In [10]:
def score_query(query, df):
    query_words = query.lower().split()
    results = []

    for index, row in df.iterrows():
        keyword_words = row["keywords"].lower().split()
        score = 0

        for word in query_words:
            if word in keyword_words:
                score += 1

        if score > 0:
            results.append((index, score))
    results.sort(key=lambda x: x[1], reverse=True)
    return results

query = input("Enter your query: ")
results = score_query(query, df)

if len(results) == 0:
    print("No matching FAQ found.")

else:
    print("\nMatching FAQs:")
    for index, score in results:
        print("\nConfidence Score:", score)
        print("Question:", df.loc[index, "question"])
        print("Answer:", df.loc[index, "answer"])
        print("Category:", df.loc[index, "category"])

Enter your query: how can i pay the fee

Matching FAQs:

Confidence Score: 2
Question: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.
Category: billing

Confidence Score: 1
Question: what is the annual fee
Answer: The annual fee is Rs 500.
Category: billing


Q3. Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1, and print the result.

In [11]:
def same_category(category_name, df):
    return df[df["category"] == category_name]

category_name = "billing"
result = same_category(category_name, df)

print("FAQs belonging to category:", category_name)
result

FAQs belonging to category: billing


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing


Q4. Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named your_roll_number_faq_data.csv.

In [13]:
index = 0

print("Selected question:")
print(df.loc[index, "question"])

new_keyword = input("Enter a new keyword: ")

df.loc[index, "keywords"] = df.loc[index, "keywords"] + " " + new_keyword

roll_num = input("Enter your roll number: ")
df.to_csv(roll_num + "_faq_data.csv", index=False)

print("\nUpdated keywords:")
print(df.loc[index, "keywords"])

print("\nDataFrame saved to", roll_num + "_faq_data.csv")

Selected question:
what is the annual fee
Enter a new keyword: charges
Enter your roll number: 1024170288

Updated keywords:
fee cost price charge fee cost charges

DataFrame saved to 1024170288_faq_data.csv


Q5. Using groupby, print how many FAQ entries you have per category.

In [6]:
category_count = df.groupby("category").size()

print("Number of FAQ entries per category:")
print(category_count)

Number of FAQ entries per category:
category
account    2
billing    2
general    1
dtype: int64


Q6. Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one — it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.

In [7]:
def score_query_with_ties(query, df):

    query_words = query.lower().split()

    scores = []

    for i in range(len(df)):

        score = 0

        keyword_words = df.loc[i, "keywords"].lower().split()

        for word in query_words:
            if word in keyword_words:
                score += 1

        scores.append((i, score))

    highest_score = max(score for i, score in scores)

    if highest_score == 0:
        print("No matching FAQ found.")
        return

    print("Highest Score:", highest_score)

    for i, score in scores:

        if score == highest_score:
            print("\nQuestion:", df.loc[i, "question"])
            print("Answer:", df.loc[i, "answer"])
            print("Category:", df.loc[i, "category"])


print("TIE EXAMPLE")
score_query_with_ties("fee", df)

print("\nNO TIE EXAMPLE")
score_query_with_ties("password", df)

TIE EXAMPLE
Highest Score: 1

Question: what is the annual fee
Answer: The annual fee is Rs 500.
Category: billing

Question: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.
Category: billing

NO TIE EXAMPLE
Highest Score: 1

Question: how to reset password
Answer: Go to Settings > Reset Password.
Category: account
